Let's build the relational dataset for finetuning

Script for finetuning an LLM.

In [103]:
import json
import pandas as pd

# List to hold flattened rows.
rows = []

# Read the JSON Lines file containing the layered summaries.
input_file = "layered_summaries_with_prompts_streamed.jsonl"
with open(input_file, "r") as infile:
    for line in infile:
        if not line.strip():
            continue
        record = json.loads(line)
        filename = record.get("filename", "")
        summaries = record.get("summaries", {})
        
        # For each section in the summaries, create a row.
        for section, summary in summaries.items():
            rows.append({
                "filename": filename,
                "section": section,
                "summary": summary
            })

# Create a DataFrame from the flattened rows.
df = pd.DataFrame(rows)

# Display the first few rows.
print("Flattened Summaries DataFrame:")
print(df.head())




Flattened Summaries DataFrame:
                                           filename  \
0  [14]20230123 HUG2 BJC Observation Report v1.1.md   
1  [14]20230123 HUG2 BJC Observation Report v1.1.md   
2  [14]20230123 HUG2 BJC Observation Report v1.1.md   
3  [14]20230123 HUG2 BJC Observation Report v1.1.md   
4  [14]20230123 HUG2 BJC Observation Report v1.1.md   

                       section  \
0    PROJECT EXECUTIVE SUMMARY   
1  PROJECT MATURITY ASSESSMENT   
2                     FINDINGS   
3              RECOMMENDATIONS   
4       RAG STATUS DEFINITIONS   

                                             summary  
0  The Home Upgrade Grant Phase 2 (HUG2) is a pro...  
1  The Home Upgrade Grant Phase 2 (HUG2) project,...  
2  | Finding/observation | Impact of Finding/obse...  
3  The text provides recommendations for the Home...  
4  The text provides RAG status definitions for a...  


In [8]:
import pandas as pd
import pandas as pd

def sort_key(filename):
    # Remove whitespace and extract the first three characters.
    key = filename.strip()[:3]
    try:
        # Convert the key to an integer if possible.
        return int(key)
    except ValueError:
        return key

# Extract unique filenames.
unique_filenames = df['filename'].unique()

# Sort them in descending order based on the sort_key.
sorted_filenames = sorted(unique_filenames, key=sort_key, reverse=True)

# Create a DataFrame from the sorted filenames.
df_filenames = pd.DataFrame(sorted_filenames, columns=['filename'])
# Optionally, add a column for the first three characters.
df_filenames['key'] = df_filenames['filename'].str.strip().str[:3]

# Display the DataFrame.
print(df_filenames)

# Assuming df_filenames is already created.
# Update the key column by stripping whitespace, extracting the first three characters,
# and then removing any square brackets.
df_filenames['key'] = df_filenames['filename'].str.strip().str[:3].str.replace(r'[\[\]]', '', regex=True)

# Optionally, if you expect the key to be numeric, you can convert it:
# df_filenames['key'] = pd.to_numeric(df_filenames['key'], errors='ignore')

# Define keywords to identify evaluation documents.
keywords = ["bcat", "assurance", "observation"]
pattern = "|".join(keywords)

# Split the DataFrame into evaluation and government docs based on filename.
df_eval = df_filenames[df_filenames["filename"].str.lower().str.contains(pattern)]
df_gov = df_filenames[~df_filenames["filename"].str.lower().str.contains(pattern)]

# Merge the two groups on the key using an outer join so we capture any missing pairs.
df_matched = pd.merge(df_gov, df_eval, on="key", how="outer", suffixes=("_gov", "_eval"))

# Identify keys with missing matches.
mismatch_mask = df_matched["filename_gov"].isnull() | df_matched["filename_eval"].isnull()
df_mismatches = df_matched[mismatch_mask]

# Now, display the results.
print("Matched documents by key:")
print(df_matched)

print("\nFilenames with missing corresponding match:")
print(df_mismatches)


                                             filename  key
0   [9]20221017 BCAT Assurance Observations Metro ...  [9]
1   [9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...  [9]
2   [8]FINAL_VLR_BCAT Assurance Observations Repor...  [8]
3   [8]wmcasaf-soc-strategic-outline-case-board-fr...  [8]
4   [7]Final Assurance Observations Report Bus Dep...  [7]
5   [6]_UK Shared Prosperity Fund BCAT Assurance O...  [6]
6   [6]WMCA-SAF-PBC - UKSPF Programme Business Cas...  [6]
7   [5]BCAT Assurance Observations Report_v02 with...  [5]
8   [5]APPROVED FINAL Coventry CCS - Full Business...  [5]
9   [4]WMCA-SAF-AEB BCAT Assurance Observations Re...  [4]
10                           [4]AEB BJC (02.10.23).md  [4]
11                [38]Dudley Port PBC Assurance_v1.md  [38
12  [37]BCAT Assurance Observations Report - Influ...  [37
13  [37]WMCA-SAF-TP018_PBC - Programme Business Ca...  [37
14  [36]Local Investment in Natural Capital (LINC)...  [36
15  [36]WMCA-SAF-TP018_PBC - Programme Business Ca...  [

In [104]:
df_matched_clean = df_matched.dropna(subset=['filename_gov', 'filename_eval'])
df_matched_clean

,filename_gov,key,filename_eval
0,[1]WMCASAF - PBC - Programme Business Case - S...,1,[1]Skills PBC BCAT V1.md
1,[10]WMCA-SAF-TP016 _FBC BEAS V4.md,10,[10]BCAT Assurance Observations Report Busines...
3,[12]issued updated doc WMCASAF - BJC - Busines...,12,[12]20230130 Final BCAT Assurance Observations...
4,[13]ATF3 PBC V3.md,13,[13]PBC ATF3 BCAT Assurance Observations Repor...
6,[15]2023 05 TP Review wmcasaf-full-business-ca...,15,[15]Final Agreed BCAT Assurance Observations R...
7,[16]CCATCTI Phase 1 SAF FBC v2.md,16,[16]CCATCIPhase1FBCBCAT Assurance Observations...
9,[18]V.2 CLEAN VERSION - GWM FBC - WMCA Submiss...,18,[18]CWG Legacy Funding FBC BCAT Assurance Obse...
10,[19]FCFJ BJC Draft.md,19,[19]Free Courses for job BCAT Assurance Observ...
11,[2]FOR SUBMISSION - WMCASAF - Housing HPR Inve...,2,[2]Housing Programme Business Case BCAT Assura...
12,[20]WMCASAF - BJC - Business Justification Cas...,20,[20]Wave 3 Bootcamp Expansion_BCAT Assurance O...


In [105]:
df_matched_clean.to_csv("df_matched_clean.csv", index=False)


In [106]:
import pandas as pd
import json

# ---------------------------------
# Step 1: Create the flattened summaries DataFrame (df_flat)
# ---------------------------------
input_file = "layered_summaries_with_prompts_streamed.jsonl"
rows = []
with open(input_file, "r") as infile:
    for line in infile:
        if line.strip():
            record = json.loads(line)
            filename = record.get("filename", "")
            summaries = record.get("summaries", {})
            # Flatten the summaries so that each section becomes a row.
            for section, summary in summaries.items():
                rows.append({
                    "filename": filename,
                    "section": section,
                    "summary": summary
                })
df_flat = pd.DataFrame(rows)

# ---------------------------------
# Step 2: Create DataFrames for government and evaluation docs
# ---------------------------------
# For government documents, rename the columns accordingly.
df_gov_flat = df_flat.copy()
df_gov_flat.rename(columns={'filename': 'filename_gov', 
                            'section': 'section_gov', 
                            'summary': 'summaries_gov'}, inplace=True)

# For evaluation documents, rename the columns.
df_eval_flat = df_flat.copy()
df_eval_flat.rename(columns={'filename': 'filename_eval', 
                             'section': 'section_eval', 
                             'summary': 'summaries_eval'}, inplace=True)

# ---------------------------------
# Step 3: Merge the flattened summaries with the matched DataFrame (df_matched_clean)
# ---------------------------------
# For government documents:
df_gov_final = pd.merge(df_matched_clean[['key', 'filename_gov']], 
                        df_gov_flat, 
                        on='filename_gov', 
                        how='left')

# For evaluation documents:
df_eval_final = pd.merge(df_matched_clean[['key', 'filename_eval']], 
                         df_eval_flat, 
                         on='filename_eval', 
                         how='left')

# ---------------------------------
# Step 4: Display the results
# ---------------------------------
print("Government Documents Summary:")
print(df_gov_final.head())

print("\nEvaluation Documents Summary:")
print(df_eval_final.head())



Government Documents Summary:
  key                                       filename_gov        section_gov  \
0   1  [1]WMCASAF - PBC - Programme Business Case - S...  EXECUTIVE SUMMARY   
1   1  [1]WMCASAF - PBC - Programme Business Case - S...     STRATEGIC CASE   
2   1  [1]WMCASAF - PBC - Programme Business Case - S...      ECONOMIC CASE   
3   1  [1]WMCASAF - PBC - Programme Business Case - S...    COMMERCIAL CASE   
4   1  [1]WMCASAF - PBC - Programme Business Case - S...     FINANCIAL CASE   

                                       summaries_gov  
0  Title: West Midlands Combined Authority (WMCA)...  
1  The West Midlands Combined Authority (WMCA) pr...  
2  The West Midlands Combined Authority (WMCA) Sk...  
3  The Commercial Case for the West Midlands Comb...  
4  The West Midlands Combined Authority (WMCA) pr...  

Evaluation Documents Summary:
  key                                      filename_eval  \
0   1                           [1]Skills PBC BCAT V1.md   
1   1         

In [107]:
df_gov_final

,key,filename_gov,section_gov,summaries_gov
0,1,[1]WMCASAF - PBC - Programme Business Case - S...,EXECUTIVE SUMMARY,Title: West Midlands Combined Authority (WMCA)...
1,1,[1]WMCASAF - PBC - Programme Business Case - S...,STRATEGIC CASE,The West Midlands Combined Authority (WMCA) pr...
2,1,[1]WMCASAF - PBC - Programme Business Case - S...,ECONOMIC CASE,The West Midlands Combined Authority (WMCA) Sk...
3,1,[1]WMCASAF - PBC - Programme Business Case - S...,COMMERCIAL CASE,The Commercial Case for the West Midlands Comb...
4,1,[1]WMCASAF - PBC - Programme Business Case - S...,FINANCIAL CASE,The West Midlands Combined Authority (WMCA) pr...
...,...,...,...,...
132,9,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,STRATEGIC CASE,The Midlands Metro Renewals Programme aims to ...
133,9,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,ECONOMIC CASE,The Economic Case for the Midlands Metro Line ...
134,9,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,COMMERCIAL CASE,The Midland Metro Alliance (MMA) is the delive...
135,9,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,FINANCIAL CASE,The text discusses the financial case for the ...


In [108]:
df_eval_final

,key,filename_eval,section_eval,summaries_eval
0,1,[1]Skills PBC BCAT V1.md,PROJECT EXECUTIVE SUMMARY,The text describes a Business Case (PBC) for t...
1,1,[1]Skills PBC BCAT V1.md,FINDINGS,The text presents findings and recommendations...
2,1,[1]Skills PBC BCAT V1.md,RECOMMENDATIONS,The text provides prioritized recommendations ...
3,1,[1]Skills PBC BCAT V1.md,RAG STATUS DEFINITIONS,The text describes a Business Case for the Ski...
4,10,[10]BCAT Assurance Observations Report Busines...,PROJECT EXECUTIVE SUMMARY,The Business Energy Assessment Service (BEAS) ...
...,...,...,...,...
101,9,[9]20221017 BCAT Assurance Observations Metro ...,PROJECT EXECUTIVE SUMMARY,The text discusses a business case for the ren...
102,9,[9]20221017 BCAT Assurance Observations Metro ...,PROJECT MATURITY ASSESSMENT,The text presents the results of a business ca...
103,9,[9]20221017 BCAT Assurance Observations Metro ...,FINDINGS,The findings and recommendations from the Busi...
104,9,[9]20221017 BCAT Assurance Observations Metro ...,RECOMMENDATIONS,The table provides recommendations based on th...


In [109]:
rag = df_eval_final[df_eval_final['section_eval'] == "RAG STATUS DEFINITIONS"]
df_eval_final = df_eval_final[df_eval_final['section_eval'] != "RAG STATUS DEFINITIONS"]


In [110]:
df_rag = rag
df_rag.rename(columns={"summaries_eval": "rag_instructions"}, inplace=True)
df_rag['rag_instructions'][8]

/var/folders/rz/68rzz4790djdsk13ycd911f00000gr/T/ipykernel_1622/176981457.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rag.rename(columns={"summaries_eval": "rag_instructions"}, inplace=True)


"The text provides RAG status definitions and prioritization of Programme Assurance Team Recommendations for the Business Energy Assessment Service (BEAS) project.\n\nRAG status definitions:\n- Green (80-100%): Successful delivery is highly likely with no major outstanding issues.\n- Amber (40-59%): Successful delivery is probable but requires constant attention to manage risks.\n- Amber/Red (20-39%): Successful delivery is in doubt with major risks or issues apparent in a number of key areas. Urgent action is needed.\n- Red (0-19%): Successful delivery appears unachievable with major issues that do not seem manageable or resolvable.\n\nPrioritization of Programme Assurance Team Recommendations:\n- High (Do Now): Essential recommendations that should be addressed immediately to increase the likelihood of a successful outcome.\n- Medium (Do By): Recommendations that should be addressed by a specified date in the near future to increase the likelihood of a successful outcome.\n- Low (Rec

In [111]:
import pandas as pd
import tiktoken

# Initialize tiktoken using GPT-2 encoding (adjust if needed)
encoding = tiktoken.get_encoding("gpt2")

def count_tokens(text):
    """Return the number of tokens in the given text."""
    return len(encoding.encode(text))

def chunk_text(text, max_tokens=500, overlap=50):
    """
    Splits text into chunks of at most max_tokens tokens with a given overlap.
    """
    tokens = encoding.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk = encoding.decode(tokens[start:end])
        chunks.append(chunk)
        if end == len(tokens):
            break
        start = end - overlap
    return chunks

def accumulate_chunks(text, max_total_tokens=2048):
    """
    Accumulates chunks from text so that the total token count is within max_total_tokens.
    If the text is already under the limit, it returns the text as-is.
    """
    if count_tokens(text) <= max_total_tokens:
        return text.strip()
    chunks = chunk_text(text, max_tokens=500, overlap=50)
    accumulated = ""
    for chunk in chunks:
        if count_tokens(accumulated + " " + chunk) <= max_total_tokens:
            accumulated += " " + chunk
        else:
            break
    return accumulated.strip()

# ---------------------------
# Step 1: Create a mapping DataFrame for the executive summaries.
# ---------------------------
exec_mapping = (
    df_gov_final[df_gov_final['section_gov'].str.upper().isin(["EXECUTIVE SUMMARY", "PROJECT EXECUTIVE SUMMARY"])]
    [["filename_gov", "summaries_gov"]]
    .rename(columns={"summaries_gov": "executive_summary"})
)

# Cap the executive_summary text to 2048 tokens.
exec_mapping["executive_summary"] = exec_mapping["executive_summary"].apply(lambda x: accumulate_chunks(x, max_total_tokens=2048))

# ---------------------------
# Step 2: Merge the mapping back into the government docs DataFrame on filename.
# ---------------------------
df_gov_prompts = pd.merge(df_gov_final, exec_mapping, on="filename_gov", how="left")

# Exclude the executive summary rows from prompt generation.
df_gov_prompts = df_gov_prompts[~df_gov_prompts['section_gov'].str.upper().isin(["EXECUTIVE SUMMARY", "PROJECT EXECUTIVE SUMMARY"])]

# Optionally, cap the individual section summaries as well.
df_gov_prompts["summaries_gov"] = df_gov_prompts["summaries_gov"].apply(lambda x: accumulate_chunks(x, max_total_tokens=2048))

# ---------------------------
# Step 3: Generate the prompt for each row.
# ---------------------------
df_gov_prompts['prompt'] = df_gov_prompts.apply(
    lambda row: (
        f"You are an expert government official writing a business case for a program summarized as "
        f"{row['executive_summary']}, write me a concise {row['section_gov']} for it. "
        f"Response: For {row['section_gov']} you could write {row['summaries_gov']}."
    ), axis=1
)

# Display the first few rows with filename, section, and generated prompt.
print(df_gov_prompts[['filename_gov', 'section_gov', 'prompt']])


                                          filename_gov      section_gov  \
1    [1]WMCASAF - PBC - Programme Business Case - S...   STRATEGIC CASE   
2    [1]WMCASAF - PBC - Programme Business Case - S...    ECONOMIC CASE   
3    [1]WMCASAF - PBC - Programme Business Case - S...  COMMERCIAL CASE   
4    [1]WMCASAF - PBC - Programme Business Case - S...   FINANCIAL CASE   
5    [1]WMCASAF - PBC - Programme Business Case - S...  MANAGEMENT CASE   
..                                                 ...              ...   
132  [9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...   STRATEGIC CASE   
133  [9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...    ECONOMIC CASE   
134  [9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...  COMMERCIAL CASE   
135  [9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...   FINANCIAL CASE   
136  [9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...  MANAGEMENT CASE   

                                                prompt  
1    You are an expert government official

In [112]:
print(df_gov_prompts['prompt'][1])


You are an expert government official writing a business case for a program summarized as Title: West Midlands Combined Authority (WMCA) Skills Programme Summary

The West Midlands Combined Authority (WMCA) proposes a comprehensive skills programme, encompassing various initiatives such as the Adult Education Budget (AEB), Free Courses for Jobs (FCFJ), Technical Bootcamps, Multiply, and UK Shared Prosperity Fund (UKSPF). Future projects may include CITB In-work Mentoring, Employment Support Delivery Pilot, extensions to the Connecting Communities programme, and a Traineeships programme.

The programme aligns with WMCA's strategic objectives and policies, particularly the Regional Skills Plan, the Plan for Growth, and the Adult Education Budget Strategy 22-27. The objectives of the skills programmes are to equip every resident with the necessary skills for employment and career progression, and to support businesses in the region to boost productivity and economic growth.

The programme

In [116]:
import tiktoken

# Initialize the tokenizer (using GPT-2 encoding as before)
encoding = tiktoken.get_encoding("gpt2")

def count_tokens(text):
    """Return the number of tokens in the given text."""
    return len(encoding.encode(text))

# Add a new column 'prompt_tokens' to df_gov_prompts with the token counts.
df_gov_prompts['prompt_tokens'] = df_gov_prompts['prompt'].apply(count_tokens)

# Display a few rows with the prompt and its token count.
df_gov_prompts = df_gov_prompts[['filename_gov', 'section_gov', 'prompt', 'prompt_tokens']]
df_gov_prompts

,filename_gov,section_gov,prompt,prompt_tokens
1,[1]WMCASAF - PBC - Programme Business Case - S...,STRATEGIC CASE,You are an expert government official writing ...,2839
2,[1]WMCASAF - PBC - Programme Business Case - S...,ECONOMIC CASE,You are an expert government official writing ...,2102
3,[1]WMCASAF - PBC - Programme Business Case - S...,COMMERCIAL CASE,You are an expert government official writing ...,1068
4,[1]WMCASAF - PBC - Programme Business Case - S...,FINANCIAL CASE,You are an expert government official writing ...,1195
5,[1]WMCASAF - PBC - Programme Business Case - S...,MANAGEMENT CASE,You are an expert government official writing ...,1640
...,...,...,...,...
132,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,STRATEGIC CASE,You are an expert government official writing ...,2422
133,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,ECONOMIC CASE,You are an expert government official writing ...,1782
134,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,COMMERCIAL CASE,You are an expert government official writing ...,2415
135,[9]FBC - MIDLAND METRO LINE ONE RENEWALS PROGR...,FINANCIAL CASE,You are an expert government official writing ...,1677


In [118]:
import pandas as pd
import tiktoken

# Assume df_gov_final and df_eval_final are already loaded.
# df_gov_final should have columns: key, filename_gov, section_gov, summaries_gov
# df_eval_final should have columns: key, filename_eval, section_eval, summaries_eval

# ---------------------------------------------------
# Step 1: Aggregate the Government Document Summaries
# ---------------------------------------------------
# Filter government sections if necessary. Here we assume all rows in df_gov_final are relevant.
gov_grouped = df_gov_final.groupby('key').agg({
    'filename_gov': 'first', 
    'summaries_gov': lambda x: " ".join(x)
}).reset_index()
gov_grouped

,key,filename_gov,summaries_gov
0,1,[1]WMCASAF - PBC - Programme Business Case - S...,Title: West Midlands Combined Authority (WMCA)...
1,10,[10]WMCA-SAF-TP016 _FBC BEAS V4.md,Title: Energy Efficiency Programme for West Mi...
2,12,[12]issued updated doc WMCASAF - BJC - Busines...,Title: West Midlands Combined Authority (WMCA)...
3,13,[13]ATF3 PBC V3.md,Title: West Midlands Cycling and Walking Progr...
4,15,[15]2023 05 TP Review wmcasaf-full-business-ca...,"Title: Wolverhampton City Centre: Walk, Cycle,..."
5,16,[16]CCATCTI Phase 1 SAF FBC v2.md,Title: City Centre Active Travel Connections t...
6,18,[18]V.2 CLEAN VERSION - GWM FBC - WMCA Submiss...,Title: Global West Midlands Programme: A Two-Y...
7,19,[19]FCFJ BJC Draft.md,Title: Summary of the Proposed Level 3 Trainin...
8,2,[2]FOR SUBMISSION - WMCASAF - Housing HPR Inve...,Title: West Midlands Combined Authority (WMCA)...
9,20,[20]WMCASAF - BJC - Business Justification Cas...,Title: West Midlands Combined Authority Techni...


In [119]:
df_eval_final

,key,filename_eval,section_eval,summaries_eval
0,1,[1]Skills PBC BCAT V1.md,PROJECT EXECUTIVE SUMMARY,The text describes a Business Case (PBC) for t...
1,1,[1]Skills PBC BCAT V1.md,FINDINGS,The text presents findings and recommendations...
2,1,[1]Skills PBC BCAT V1.md,RECOMMENDATIONS,The text provides prioritized recommendations ...
4,10,[10]BCAT Assurance Observations Report Busines...,PROJECT EXECUTIVE SUMMARY,The Business Energy Assessment Service (BEAS) ...
5,10,[10]BCAT Assurance Observations Report Busines...,PROJECT MATURITY ASSESSMENT,The Business Energy Assessment Service (BEAS) ...
...,...,...,...,...
99,8,[8]FINAL_VLR_BCAT Assurance Observations Repor...,RECOMMENDATIONS,"The table summarizes the findings, recommendat..."
101,9,[9]20221017 BCAT Assurance Observations Metro ...,PROJECT EXECUTIVE SUMMARY,The text discusses a business case for the ren...
102,9,[9]20221017 BCAT Assurance Observations Metro ...,PROJECT MATURITY ASSESSMENT,The text presents the results of a business ca...
103,9,[9]20221017 BCAT Assurance Observations Metro ...,FINDINGS,The findings and recommendations from the Busi...


In [ ]:
import pandas as pd
from mlx_lm import generate, load

# Load the Mistral model (adjust path/configuration accordingly)
checkpoint = "scripts/llm/mlx_models/Mistral-7B-Instruct-v0.3"
model, tokenizer = load(path_or_hf_repo=checkpoint)

# Define helper functions that use the Mistral tokenizer.
def count_tokens(text):
    """Return the number of tokens in the given text using the Mistral tokenizer."""
    return len(tokenizer.encode(text))

def chunk_text(text, max_tokens=500, overlap=50):
    """
    Splits text into chunks of at most max_tokens tokens with a given overlap using the Mistral tokenizer.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk = tokenizer.decode(tokens[start:end])
        chunks.append(chunk)
        if end == len(tokens):
            break
        start = end - overlap  # slide back by the overlap tokens
    return chunks

def accumulate_chunks(text, max_total_tokens=1024):
    """
    Accumulates chunks from the text so that the total token count is within max_total_tokens.
    If the text is already under the limit, it returns the text as-is.
    """
    if count_tokens(text) <= max_total_tokens:
        return text.strip()
    chunks = chunk_text(text, max_tokens=500, overlap=50)
    accumulated = ""
    for chunk in chunks:
        if count_tokens(accumulated + " " + chunk) <= max_total_tokens:
            accumulated += " " + chunk
        else:
            break
    return accumulated.strip()

# --------------------------------------
# Step 1: Group and Concatenate Evaluation Summaries
# --------------------------------------
# Assume df_eval_final is your cleaned DataFrame with evaluation documents.
# It should have at least these columns: "key", "filename_eval", "section_eval", "summaries_eval".

# Group by key and concatenate all the evaluation summaries for that project.
df_eval_grouped = df_eval_final.groupby("key").agg({
    "filename_eval": "first",  # use the first filename as representative
    "summaries_eval": lambda x: " ".join(x)
}).reset_index()

# --------------------------------------
# Step 2: Apply Chunking to Ensure the Combined Text is Below the Token Limit
# --------------------------------------
# Use the accumulate_chunks function (with the Mistral tokenizer) on the concatenated evaluation summaries.
df_eval_grouped["combined_eval_summary"] = df_eval_grouped["summaries_eval"].apply(
    lambda x: accumulate_chunks(x, max_total_tokens=1024)
)

# Optionally, count tokens for inspection.
df_eval_grouped["combined_tokens"] = df_eval_grouped["combined_eval_summary"].apply(count_tokens)

# Display the resulting DataFrame.
print(df_eval_grouped[["key", "filename_eval", "combined_eval_summary", "combined_tokens"]])


/Users/abeltran/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   key                                      filename_eval  \
0    1                           [1]Skills PBC BCAT V1.md   
1   10  [10]BCAT Assurance Observations Report Busines...   
2   12  [12]20230130 Final BCAT Assurance Observations...   
3   13  [13]PBC ATF3 BCAT Assurance Observations Repor...   
4   15  [15]Final Agreed BCAT Assurance Observations R...   
5   16  [16]CCATCIPhase1FBCBCAT Assurance Observations...   
6   18  [18]CWG Legacy Funding FBC BCAT Assurance Obse...   
7   19  [19]Free Courses for job BCAT Assurance Observ...   
8    2  [2]Housing Programme Business Case BCAT Assura...   
9   20  [20]Wave 3 Bootcamp Expansion_BCAT Assurance O...   
10  21  [21]FINAL CWGLF Inclusive Communities Grant_BC...   
11  24  [24]Stratford Gateway Final Agreed BCAT Assura...   
12  26  [26]BCAT Assurance Observations Report Social ...   
13  28  [28]Assest V1.0 Capital Renewal BCAT Assurance...   
14  31  [31]CWG Major Events PBC BCAT Assurance Observ...   
15  34  [34]FINAL_CWGLF 

In [68]:
df_eval_grouped['combined_eval_summary'][0]

"The text describes a Business Case (PBC) for the Skills Programme, a project led by Clare Hatton in the West Midlands Combined Authority (WMCA). The Skills Programme aims to provide various training and education opportunities, including Adult Education Budget (AEB), Free Courses for Jobs (FCFJ), Technical Bootcamps, Multiply, and UK Shared Prosperity Fund (UKSPF). Future projects include CITB In-work Mentoring, Employment Support Delivery Pilot, extensions to the Connecting Communities programme, and a Traineeships programme.\n\nThe Skills Team intends to commission provision collectively to simplify arrangements for training providers, reduce duplication, and maximize impact. Funding for the Skills Programme has already been secured, including devolved AEB, delegated National Skills Fund 'Free Courses for Jobs' and 'Technical Bootcamps', delegated UKSPF and Multiply funding grant.\n\nThe Business Case Assessment Tool (BCAT) was used to evaluate the maturity of the Skills Programme B

In [ ]:
import pandas as pd
from mlx_lm import generate, load

# -----------------------------
# Load the Mistral model and tokenizer
# -----------------------------
checkpoint = "scripts/llm/mlx_models/Mistral-7B-Instruct-v0.3"
model, tokenizer = load(path_or_hf_repo=checkpoint)

# -----------------------------
# Helper functions for tokenization and chunking using the Mistral tokenizer
# -----------------------------
def count_tokens(text):
    """Return the number of tokens in the given text using the Mistral tokenizer."""
    return len(tokenizer.encode(text))

def chunk_text(text, max_tokens=500, overlap=50):
    """
    Splits text into chunks of at most max_tokens tokens with a given overlap.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk = tokenizer.decode(tokens[start:end])
        chunks.append(chunk)
        if end == len(tokens):
            break
        start = end - overlap
    return chunks

def accumulate_chunks(text, max_total_tokens=1000):
    """
    Accumulates chunks from text so that the total token count is within max_total_tokens.
    If the text is already under the limit, returns the text as-is.
    """
    if count_tokens(text) <= max_total_tokens:
        return text.strip()
    chunks = chunk_text(text, max_tokens=500, overlap=50)
    accumulated = ""
    for chunk in chunks:
        if count_tokens(accumulated + " " + chunk) <= max_total_tokens:
            accumulated += " " + chunk
        else:
            break
    return accumulated.strip()

# -----------------------------
# Step 1: Aggregate Government Document Summaries
# -----------------------------
# Assume df_gov_final has columns: 'key', 'filename_gov', and 'summaries_gov' (section-level summaries).
df_gov_grouped = df_gov_final.groupby('key').agg({
    'filename_gov': 'first',
    'summaries_gov': lambda x: " ".join(x)
}).reset_index()

# Limit the aggregated government text to 1000 tokens.
df_gov_grouped['aggregated_gov'] = df_gov_grouped['summaries_gov'].apply(lambda x: accumulate_chunks(x, max_total_tokens=1000))

# -----------------------------
# Step 2: Combine Evaluation Document Summaries
# -----------------------------
# Assume df_eval_final has columns: 'key', 'filename_eval', 'summaries_eval'
df_eval_grouped = df_eval_final.groupby("key").agg({
    "filename_eval": "first",
    "summaries_eval": lambda x: " ".join(x)
}).reset_index()

# Limit the combined evaluation text to 1000 tokens.
df_eval_grouped["combined_eval_summary"] = df_eval_grouped["summaries_eval"].apply(lambda x: accumulate_chunks(x, max_total_tokens=1000))

# -----------------------------
# Step 3: Merge with Unique RAG Instructions
# -----------------------------
# Assume df_rag has columns: 'key' and 'rag_instructions' (unique per project).
# Normalize keys for consistent merging.
df_rag['key'] = df_rag['key'].astype(str).str.strip().str.upper()
df_gov_grouped['key'] = df_gov_grouped['key'].astype(str).str.strip().str.upper()
df_eval_grouped['key'] = df_eval_grouped['key'].astype(str).str.strip().str.upper()

# Merge government, evaluation, and RAG instructions by key.
df_merged_all = df_gov_grouped.merge(df_eval_grouped[['key', 'filename_eval', 'combined_eval_summary']], on='key', how='inner')
df_merged_all = df_merged_all.merge(df_rag[['key', 'rag_instructions']], on='key', how='inner')

# -----------------------------
# Step 4: Build Training Examples
# -----------------------------
training_examples = []
for idx, row in df_merged_all.iterrows():
    gov_text = row['aggregated_gov']
    rag_inst = row['rag_instructions']
    eval_text = row['combined_eval_summary']
    
    input_prompt = (
        f"You are an expert evaluator. The RAG definitions are summarized as: {rag_inst} "
        f"Here is the aggregated government business case for the project: {gov_text} "
        "Please provide a comprehensive project maturity assessment."
    )
    
    training_examples.append({
        "key": row["key"],
        "filename_gov": row["filename_gov"],
        "filename_eval": row["filename_eval"],
        "input_prompt": input_prompt,
        "target_output": eval_text
    })

# Create a DataFrame from the training examples.
df_training_final = pd.DataFrame(training_examples)

# Display the first few training examples.
print(df_training_final[['key', 'filename_gov', 'input_prompt', 'target_output']].head())


  key                                       filename_gov  \
0   1  [1]WMCASAF - PBC - Programme Business Case - S...   
1  10                [10]WMCA-SAF-TP016 _FBC  BEAS V4.md   
2  12  [12]issued updated doc WMCASAF - BJC - Busines...   
3  13                               [13]ATF3  PBC  V3.md   
4  15  [15]2023 05 TP Review wmcasaf-full-business-ca...   

                                        input_prompt  \
0  You are an expert evaluator. The RAG definitio...   
1  You are an expert evaluator. The RAG definitio...   
2  You are an expert evaluator. The RAG definitio...   
3  You are an expert evaluator. The RAG definitio...   
4  You are an expert evaluator. The RAG definitio...   

                                       target_output  
0  <s> The text describes a Business Case (PBC) f...  
1  <s> The Business Energy Assessment Service (BE...  
2  <s> The text describes a proposed project call...  
3  <s> The text describes a business case for a C...  
4  <s> The text describes a

/var/folders/rz/68rzz4790djdsk13ycd911f00000gr/T/ipykernel_1622/138473056.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rag['key'] = df_rag['key'].astype(str).str.strip().str.upper()


In [152]:
df_training_final['combined_prompt'] = df_training_final.apply(
    lambda row: f"Prompt: {row['input_prompt']}\nResponse: {row['target_output']}", axis=1
)


# Count the tokens in each combined prompt using the count_tokens function
df_training_final['token_count'] = df_training_final['combined_prompt'].apply(lambda x: count_tokens(x))

# Rename the 'combined_prompt' column to 'text'
df_training_final.rename(columns={'combined_prompt': 'text'}, inplace=True)

# Display the first few rows to verify the changes
df_training_final[['key', 'text', 'token_count']]


,key,text,token_count
0,1,Prompt: You are an expert evaluator. The RAG d...,1300
1,10,Prompt: You are an expert evaluator. The RAG d...,1336
2,12,Prompt: You are an expert evaluator. The RAG d...,1548
3,13,Prompt: You are an expert evaluator. The RAG d...,1360
4,15,Prompt: You are an expert evaluator. The RAG d...,1434
5,16,Prompt: You are an expert evaluator. The RAG d...,1683
6,18,Prompt: You are an expert evaluator. The RAG d...,1825
7,19,Prompt: You are an expert evaluator. The RAG d...,1379
8,2,Prompt: You are an expert evaluator. The RAG d...,1418
9,20,Prompt: You are an expert evaluator. The RAG d...,1388


In [179]:
df_training_final['text'][0]

"Prompt: You are an expert evaluator. The RAG definitions are summarized as: The text describes a Business Case for the Skills Programme, a project led by Clare Hatton in the West Midlands Combined Authority (WMCA). The Skills Programme aims to provide various training and education opportunities. The Business Case has been rated 'Amber' due to several high-priority recommendations, including the need for more evidence of stakeholder engagement, a benefits realisation plan, and SMART critical success factors.\n\nThe RAG status definitions provided indicate that an 'Amber' rating means the project/programme has significant issues that need to be addressed as a priority. The Programme Assurance Team's recommendations are prioritized as High (Do Now), Medium (Do By), and Low (Recommended).\n\nThe text also mentions that the Business Case will be subject to further review, should change or material updates occur. It is recommended that the Business Case is maintained as a live strategic ma

In [153]:
# Rename "prompt" to "text" in df_gov_prompts
df_gov_prompts.rename(columns={"prompt": "text"}, inplace=True)

# If you had a separate DataFrame (e.g., df_training_final) that you wanted to stack with,
# you could do:
df_full_training = pd.concat([df_gov_prompts, df_training_final], ignore_index=True)
# But since you clarified it's df_gov_prompts, we will use that as the full training dataset.


# Display a sample of the full training data
print(df_full_training[['filename_gov', 'section_gov', 'text']].head())

# Optionally, save the full training dataset to a CSV file.
df_full_training.to_csv("full_training_dataset.csv", index=False)
print("Full training dataset saved to full_training_dataset.csv")


                                        filename_gov      section_gov  \
0  [1]WMCASAF - PBC - Programme Business Case - S...   STRATEGIC CASE   
1  [1]WMCASAF - PBC - Programme Business Case - S...    ECONOMIC CASE   
2  [1]WMCASAF - PBC - Programme Business Case - S...  COMMERCIAL CASE   
3  [1]WMCASAF - PBC - Programme Business Case - S...   FINANCIAL CASE   
4  [1]WMCASAF - PBC - Programme Business Case - S...  MANAGEMENT CASE   

                                                text  
0  You are an expert government official writing ...  
1  You are an expert government official writing ...  
2  You are an expert government official writing ...  
3  You are an expert government official writing ...  
4  You are an expert government official writing ...  
Full training dataset saved to full_training_dataset.csv


In [154]:
data = df_full_training[['filename_gov', 'key', 'text']]
data

,filename_gov,key,text
0,[1]WMCASAF - PBC - Programme Business Case - S...,NaN,You are an expert government official writing ...
1,[1]WMCASAF - PBC - Programme Business Case - S...,NaN,You are an expert government official writing ...
2,[1]WMCASAF - PBC - Programme Business Case - S...,NaN,You are an expert government official writing ...
3,[1]WMCASAF - PBC - Programme Business Case - S...,NaN,You are an expert government official writing ...
4,[1]WMCASAF - PBC - Programme Business Case - S...,NaN,You are an expert government official writing ...
...,...,...,...
133,[4]AEB BJC (02.10.23).md,4,Prompt: You are an expert evaluator. The RAG d...
134,[5]APPROVED FINAL Coventry CCS - Full Business...,5,Prompt: You are an expert evaluator. The RAG d...
135,[6]WMCA-SAF-PBC - UKSPF Programme Business Cas...,6,Prompt: You are an expert evaluator. The RAG d...
136,[8]wmcasaf-soc-strategic-outline-case-board-fr...,8,Prompt: You are an expert evaluator. The RAG d...


In [178]:
import gc
gc.collect()

0

In [156]:
import pandas as pd

# Shuffle the full training dataset (set a random_state for reproducibility)
df_shuffled = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Determine the split sizes.
n = len(df_shuffled)
train_end = int(0.8 * n)
valid_end = train_end + int(0.1 * n)

# Split the data.
df_train = df_shuffled.iloc[:train_end]
df_valid = df_shuffled.iloc[train_end:valid_end]
df_test = df_shuffled.iloc[valid_end:]

# Save each split as a JSON Lines file.
df_train.to_json("train.jsonl", orient="records", lines=True)
df_valid.to_json("valid.jsonl", orient="records", lines=True)
df_test.to_json("test.jsonl", orient="records", lines=True)

print(f"Data splits saved:\n- Train: {len(df_train)} records\n- Valid: {len(df_valid)} records\n- Test: {len(df_test)} records")


Data splits saved:
- Train: 110 records
- Valid: 13 records
- Test: 15 records


In [ ]:
!mlx_lm.lora \
    --model mlx_models/Mistral-7B-Instruct-v0.3 \
    --train \
    --data llm_data \
    --iters 100

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading pretrained model
Loading datasets
Training
Trainable parameters: 0.024% (1.704M/7248.024M)
Starting training..., iters: 100
[WARNING] Some sequences are longer than 2048 tokens. The longest sentence 2360 will be truncated to 2048. Consider pre-splitting your data to save memory.
Iter 1: Val loss 1.826, Val took 79.536s
[WARNING] Some sequences are longer than 2048 tokens. The longest sentence 3133 will be truncated to 2048. Consider pre-splitting your data to save memory.
[WARNING] Some sequences are longer than 2048 tokens. The longest sentence 2084 will be truncated to 2048. Consider pre-splitting your data to save memory.
[WARNING] Some sequences are longer than 2048 tokens. The longest sentence 2982 will be truncated to 2048. Consider pre-splitting your data to save memory.
[WARNING] Some sequences are longer than 2048 tokens. The longest sentence 2343 will be truncated to 2048. Consider pre-splitting your data to save memory.
[WARNING] Some sequences are longer than 2048 t

In [ ]:
!mlx_lm.lora \
    --model mlx_models/Mistral-7B-Instruct-v0.3 \
    --adapter-path adapters \
    --data llm_data \
    --test

Python(22069) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading pretrained model
Loading datasets
Testing
[WARNING] Some sequences are longer than 2048 tokens. The longest sentence 2453 will be truncated to 2048. Consider pre-splitting your data to save memory.
Test loss 1.316, Test ppl 3.728.


In [ ]:
!pip3 install --upgrade mlx

In [159]:
!mlx_lm.generate \
    --model mlx_models/Mistral-7B-Instruct-v0.3 \
    --max-tokens 2000 \
    --adapter-path adapters \
    --prompt "You are an expert government official, write a concisce excutive summary about a bike share program."


Python(22100) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


mx.metal.set_wired_limt is deprecated and will be removed in a future version. Use mx.set_wired_limit instead.
mx.metal.get_peak_memory is deprecated and will be removed in a future version. Use mx.get_peak_memory instead.
Titlemx.metal.clear_cache is deprecated and will be removed in a future version. Use mx.clear_cache instead.
: Executive Summary for a Bike Share Program

The West Midlands Combined Authority (WMCA) is proposing a comprehensive bike share program, aimed at promoting active travel, reducing congestion and air pollution, and improving the health and wellbeing of residents and visitors. The program, titled 'BikeWest', will be a key component of the WMCA's Active Travel Plan.

BikeWest will be a self-service bike share scheme, with bikes available for short-term rental through a smartphone app. The program will initially launch with 1000 bikes and 100 stations, with plans for expansion. The bikes will be located at stations across the region, with a focus on key transpor

In [162]:
!mlx_lm.fuse \
    --model mlx_models/Mistral-7B-Instruct-v0.3 \
    --adapter-path adapters \
    --save-path models/wmca_mistral_v2 \
    --de-quantize

Python(28322) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading pretrained model
De-quantizing model


In [ ]:
!cd scripts/llm/llama.cpp
!pip install -r llama.cpp/requirements.txt
!pip install torchvision 

In [170]:
!conda uninstall tensorflow
!pip3 install tensorflow==2.0.0 

Python(33631) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Solving environment: failed

PackagesNotFoundError: The following packages are missing from the target environment:
  - tensorflow




Python(33656) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
ERROR: Could not find a version that satisfies the requirement tensorflow==2.0.0 (from versions: 2.13.0rc0, 2.13.0rc1, 2.13.0rc2, 2.13.0, 2.13.1, 2.14.0rc0, 2.14.0rc1, 2.14.0, 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.1, 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0)
ERROR: No matching distribution found for tensorflow==2.0.0
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [174]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py scripts/llm/models/wmca_mistral_v2 \
     --outfile scripts/llm/models/wmca_mistral_v2.gguf \
     --outtype q8_0
     